# PANNs in Colab
Run the cells in order with a Colab GPU. If this runtime ran an earlier version of the notebook, restart it first so Python imports the updated project code. The first cells load the pretrained CNN14; later cells extract ESC-50 embeddings and train a new 50-class head on the five official folds. Large files stay in Google Drive.


In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

from google.colab import drive

project_dir = Path('/content/soundview')
panns_source_dir = Path('/content/audioset_tagging_cnn')

if project_dir.exists():
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AndreuLopezz/12607559_SoundView-environmental-sound-recognition.git', str(project_dir)], check=True)
if not panns_source_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/qiuqiangkong/audioset_tagging_cnn.git', str(panns_source_dir)], check=True)

subprocess.run(['git', '-C', str(panns_source_dir), 'checkout', 'd2f4b8c18eab44737fcc0de1248ae21eb43f6aa4'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(project_dir / 'requirements-panns.txt')], check=True)
sys.path.insert(0, str(project_dir))
drive.mount('/content/drive')
source_commit = subprocess.run(['git', '-C', str(panns_source_dir), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {importlib.metadata.version("torch")}')
print(f'TorchLibrosa: {importlib.metadata.version("torchlibrosa")}')
print(f'Official PANNs source commit: {source_commit}')

In [ ]:
import urllib.request

checkpoint_path = Path('/content/drive/MyDrive/SoundView/pretrained/Cnn14_mAP=0.431.pth')
checkpoint_url = 'https://zenodo.org/records/3987831/files/Cnn14_mAP=0.431.pth?download=1'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# Keep an incomplete download separate from the checkpoint used for experiments.
if not checkpoint_path.exists():
    temporary_path = checkpoint_path.with_suffix('.part')
    with urllib.request.urlopen(checkpoint_url, timeout=120) as source, temporary_path.open('wb') as target:
        downloaded = 0
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            downloaded += len(chunk)
            if downloaded // (50 * 1024 * 1024) != (downloaded - len(chunk)) // (50 * 1024 * 1024):
                print(f'Downloaded {downloaded / (1024 * 1024):.0f} MB')
    temporary_path.replace(checkpoint_path)

if checkpoint_path.stat().st_size < 100_000_000:
    raise ValueError('The downloaded CNN14 checkpoint is unexpectedly small')
print(f'Checkpoint ready in Drive: {checkpoint_path}')

In [ ]:
import numpy as np
import torch

from soundview.panns import load_panns_model, prepare_panns_audio

if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before loading PANNs')
device = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

# A five-second tone checks the model input and output without touching test data.
time = np.arange(44100 * 5, dtype=np.float32) / 44100
audio = 0.1 * np.sin(2 * np.pi * 440 * time)
waveform = torch.from_numpy(prepare_panns_audio(audio)).unsqueeze(0).to(device)
model = load_panns_model(panns_source_dir, checkpoint_path, device)
with torch.no_grad():
    output = model(waveform)

print(f"Embedding shape: {tuple(output['embedding'].shape)}")
print(f"AudioSet scores shape: {tuple(output['clipwise_output'].shape)}")
assert output['embedding'].shape == (1, 2048)
assert output['clipwise_output'].shape == (1, 527)

## Extract ESC-50 embeddings
CNN14 stays frozen in this experiment. Its 2,048-value representation is prepared once for every clip and cached in Drive. The audio is resampled from 44.1 to 32 kHz only for PANNs.


In [ ]:
from soundview.panns import extract_panns_embeddings
from soundview.train_panns import load_panns_features

data_dir = Path('/content/datasets/ESC-50')
if not data_dir.exists():
    data_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/karolpiczak/ESC-50.git', str(data_dir)], check=True)

RUNS_DIR = Path('/content/drive/MyDrive/SoundView/panns_frozen_v1')
FEATURES_PATH = RUNS_DIR / 'features_esc50.npz'
FEATURE_BATCH_SIZE = 4

if not FEATURES_PATH.exists():
    details = extract_panns_embeddings(
        model, data_dir, FEATURES_PATH, device, batch_size=FEATURE_BATCH_SIZE
    )
    print(f"Prepared {details['clips']} embeddings in {details['elapsed_seconds']} seconds")

features, filenames, folds, targets, class_names = load_panns_features(data_dir, FEATURES_PATH)
print(f'Feature cache ready: {features.shape}, {len(class_names)} ESC-50 classes')


## Train the frozen PANNs head
Each run trains on three folds, selects an epoch on the following validation fold, and evaluates one held-out test fold. The pretrained CNN14 is not updated. Keep these settings fixed once test results have been viewed.


In [ ]:
import json

from soundview.train_panns import train_frozen_panns_fold

MAX_EPOCHS = 100
HEAD_BATCH_SIZE = 64
LEARNING_RATE = 0.001
PATIENCE = 10
SEED = 42

expected_settings = {
    'epochs': MAX_EPOCHS,
    'batch_size': HEAD_BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'patience': PATIENCE,
    'seed': SEED,
}

for fold in range(1, 6):
    fold_dir = RUNS_DIR / f'fold{fold}'
    report_path = fold_dir / 'test_evaluation.json'
    if report_path.exists():
        training_path = fold_dir / 'training_summary.json'
        checkpoint_path = fold_dir / 'best_head.pt'
        if not training_path.exists() or not checkpoint_path.exists():
            raise FileNotFoundError(f'Fold {fold} has an incomplete saved run')
        previous = json.loads(training_path.read_text(encoding='utf-8'))
        report = json.loads(report_path.read_text(encoding='utf-8'))
        if previous['settings'] != expected_settings or previous['test_fold_reserved'] != fold:
            raise ValueError(f'Fold {fold} used different training settings')
        if report['test_fold'] != fold or report['checkpoint_epoch'] != previous['best_epoch']:
            raise ValueError(f'Fold {fold} report does not match its checkpoint')
        print(f'Fold {fold} already complete; reusing its saved report.')
        continue

    train_frozen_panns_fold(
        data_dir=data_dir,
        features_path=FEATURES_PATH,
        output_dir=fold_dir,
        test_fold=fold,
        epochs=MAX_EPOCHS,
        batch_size=HEAD_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        patience=PATIENCE,
        seed=SEED,
    )


In [ ]:
from soundview.fivefold import combine_fold_reports

report_paths = [RUNS_DIR / f'fold{fold}' / 'test_evaluation.json' for fold in range(1, 6)]
summary_path = RUNS_DIR / 'fivefold_summary.json'
summary = combine_fold_reports(report_paths, summary_path)

for fold in summary['folds']:
    print(f"Fold {fold['test_fold']}: accuracy {fold['accuracy']:.4f}, macro-F1 {fold['macro_f1']:.4f}")
print(f"Mean fold macro-F1: {summary['mean_fold_macro_f1']:.4f}")
print(f"Pooled macro-F1 across 2,000 clips: {summary['pooled']['macro_f1']:.4f}")
print(f'Report saved to {summary_path}')


## Measure partial fine-tuning on the T4
The frozen-head result is already saved. This short trial starts from its fold-5 head, unlocks only CNN14's last convolution block and embedding layer, and performs a few updates using training-fold clips. It does not score the test fold or save a new model. The time estimate covers training batches; validation adds more time.


In [ ]:
import json

from soundview.finetune_panns import benchmark_partial_finetuning

# Release the frozen copy before loading a trainable CNN14 on the T4.
if 'model' in globals():
    del model
torch.cuda.empty_cache()

benchmark = benchmark_partial_finetuning(
    data_dir=data_dir,
    source_dir=panns_source_dir,
    pretrained_checkpoint=checkpoint_path,
    head_checkpoint=RUNS_DIR / 'fold5' / 'best_head.pt',
    test_fold=5,
    batch_size=2,
    measured_batches=3,
)
benchmark_path = RUNS_DIR / 'partial_trial_benchmark.json'
benchmark_path.write_text(json.dumps(benchmark, indent=2), encoding='utf-8')
print(json.dumps(benchmark, indent=2))
print(f'Benchmark saved to {benchmark_path}')
